# Document Scanner — Kaggle GPU Corner Detection Training Launcher

Runs the corner detection comparison between Approach A and Approach B (`[REQ-30]`, `[REQ-31]`, ADR-007) on a Kaggle GPU (T4 / P100):

| Run | Model | Formulation | Loss | Target |
|---|---|---|---|---|
| exp-009 | `CornerRegNet` | Approach A: Direct Coordinate Regression | L1 | 8 normalized coordinates in [0, 1] |
| exp-010 | `CornerHeatmapNet` | Approach B: Heatmap Regression | MSE | 4-channel 512x512 Gaussians (sigma=8) |

### Kaggle GPU Advantage
- Kaggle provides **30 hours/week** of free GPU compute (NVIDIA P100 / T4 x2).
- Persistent output directory at `/kaggle/working/` allowing direct zip download of checkpoints and figures.

### Step 0: Confirm GPU & Environment Setup

In [ ]:
import os, sys, torch
print('PyTorch version:', torch.__version__)
print('vCPUs:', os.cpu_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('WARNING: CUDA GPU not detected. Ensure Accelerator is set to GPU T4 x2 or P100 in Kaggle settings.')

### Step 1: Clone Repository & Change Working Directory

In [ ]:
import os
repo_dir = '/kaggle/working/DocEn'
if not os.path.exists(repo_dir):
    !git clone https://github.com/HedieTahmouresi/DocEn.git {repo_dir}
%cd {repo_dir}
!git pull
!git log --oneline -1

### Step 2: Install Dependencies & Download / Extract Data

**Fast Direct Download Option**: Paste your Google Drive shareable file ID for `data.zip` in `GDRIVE_FILE_ID` below to download directly in 5-10 seconds inside Kaggle!

In [ ]:
!pip install -q -r requirements.txt gdown

import os, glob, shutil

# Optional: Set your Google Drive file ID for data.zip here
GDRIVE_FILE_ID = ""

if not os.path.exists('data/clean_scans'):
    if GDRIVE_FILE_ID:
        print('Downloading data.zip directly from Google Drive via gdown...')
        !gdown --id "{GDRIVE_FILE_ID}" -O data.zip
        !unzip -q data.zip -d .
    else:
        # Search for data.zip in kaggle input directories
        zip_candidates = glob.glob('/kaggle/input/**/data.zip', recursive=True) + ['/kaggle/working/data.zip']
        if zip_candidates:
            src_zip = zip_candidates[0]
            print('Extracting data from:', src_zip)
            !unzip -q "{src_zip}" -d .
        else:
            clean_scans_candidates = glob.glob('/kaggle/input/**/clean_scans', recursive=True)
            if clean_scans_candidates:
                data_root = os.path.dirname(clean_scans_candidates[0])
                print('Found unzipped data at:', data_root)
                !cp -r "{data_root}"/* data/
            else:
                print('WARNING: Provide GDRIVE_FILE_ID above or attach data.zip as a Kaggle dataset.')

if not os.path.exists('data/frozen/val'):
    print('Frozen evaluation sets missing - generating them now...')
    !python -m src.data.freeze

### Step 3: Run Sanity Unit Tests

In [ ]:
!python -m pytest tests/test_corner_pipeline.py -v

### Step 4: Quick 1-Epoch Smoke Run

Verifies data loading, GPU forward/backward passes, and metric logging.

In [ ]:
!python train_corners.py --env colab_t4 --epochs 1 --samples-per-epoch 100 --allow-cpu-fallback

### Step 5: Full Paired Corner Detection Training (40 Epochs)

Executes paired training of `exp-009_corner_approach_a` (RegNet) and `exp-010_corner_approach_b` (HeatmapNet) side-by-side over 40 epochs.

In [ ]:
!python train_corners.py --env colab_t4

### Step 6: Evaluate Checkpoints & Generate Report

In [ ]:
!python -m scripts.evaluate_corners

### Step 7: Generate & Display Sample Corner Overlay Visualizations

Renders predicted corner keypoints (solid dots) and ground-truth corner annotations (hollow circles) on real smartphone photos per conventions §8 (TL Red, TR Green, BR Blue, BL Yellow).

In [ ]:
!python -m scripts.save_corner_samples

import glob
from IPython.display import Image, display

sample_files = sorted(glob.glob('outputs/figures/p06_corner_overlays/*.png'))
print(f'Displaying {min(5, len(sample_files))} sample corner overlays:')
for path in sample_files[:5]:
    print('Sample:', os.path.basename(path))
    display(Image(filename=path, width=600))

### Step 8: Zip Output Results for Easy Download

Compresses `runs/` and `outputs/` into a downloadable zip file in `/kaggle/working/`.

In [ ]:
!zip -r /kaggle/working/corner_results.zip runs/ outputs/